# YOLO11s-RECSRA Reproduction Notebook
This notebook verifies the frozen package, evaluates the clean test set, and runs the ten-family mobile corruption benchmark. Existing results were generated on RTX 4090; the notebook records the actual runtime of any new execution.

In [ ]:
from pathlib import Path
import os, sys, subprocess

def locate_repo(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').is_file() and (candidate / 'scripts').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the extracted repository.')

ROOT = locate_repo(Path.cwd().resolve())
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
DATASET_ROOT = Path(os.environ.get('DATASET_ROOT', '/absolute/path/to/shrimp_dataset')).expanduser()
print('Repository:', ROOT)
print('Dataset root:', DATASET_ROOT)


## 1. Verify frozen evidence

In [ ]:
subprocess.run([sys.executable, 'scripts/verify_repository.py'], check=True, env=os.environ.copy())
subprocess.run([sys.executable, 'scripts/verify_checkpoints.py'], check=True, env=os.environ.copy())


## 2. Patch the pinned Ultralytics installation

In [ ]:
subprocess.run([sys.executable, 'scripts/patch_ultralytics.py'], check=True, env=os.environ.copy())


## 3. Clean test evaluation
Set `DATASET_ROOT` to the authorized dataset folder before running.

In [ ]:
assert (DATASET_ROOT / 'data.yaml').is_file(), DATASET_ROOT
subprocess.run([
    sys.executable, 'scripts/evaluate_clean.py',
    '--data', str(DATASET_ROOT / 'data.yaml'),
    '--device', '0',
    '--output', 'outputs/clean_test',
], check=True, env=os.environ.copy())


## 4. Mobile corruption benchmark (10 families × 5 severities)

In [ ]:
subprocess.run([
    sys.executable, 'scripts/robustness/benchmark_mobile10.py',
    '--baseline', 'checkpoints/yolo11s_baseline_best.pt',
    '--recsra', 'checkpoints/yolo11s_recsra_best.pt',
    '--dataset', str(DATASET_ROOT),
    '--output', 'outputs/mobile10',
    '--device', '0',
], check=True, env=os.environ.copy())


## 5. Review frozen percentage tables

In [ ]:
import pandas as pd
display(pd.read_csv('results/tables/clean_test_percent.csv'))
display(pd.read_csv('results/tables/top5_corruptions_percent.csv'))
